---
# Clasificador Multiclase TADC

**Diplomado Superior RNA y Deep Learning — UAEM**
**Modulo 4: Deep Learning | TADC_Clasificador**

### Que estamos construyendo?

Un clasificador que mira una foto de un personaje de
The Amazing Digital Circus y dice:
"?Esto es Pomni? ?O Jax? ?O Ragatha?"

No es solo un juguete. La misma tecnica se usa para:
- **Diagnostico medico**: ?que tipo de cancer? (multiclase)
- **Robotica**: ?que objeto tengo enfrente?
- **Moderacion**: ?que tipo de contenido?

### Diferencias con clasificacion binaria

| Aspecto | Binario (2 clases) | Multiclase (N clases) |
|---------|-------------------|----------------------|
| Salida | 1 neurona (0/1) | N neuronas (una por clase) |
| Loss | BCELoss o CrossEntropy | CrossEntropy (N clases) |
| Metrica | Accuracy, Precision, Recall | Accuracy + accuracy por clase |
| Dificultad | Separar 2 grupos | Separar N grupos simultaneamente |

Este notebook es uno de los **2 entregables** de la tarea.


In [ ]:
# ============================================
# Setup: imports, plataforma, hiperparametros
#
# ?Que hace: Importa herramientas, detecta
#   Colab vs local, define CONFIG.
#
# ?Variables:
#   - EN_COLAB (bool): True si estamos en Google Colab
#   - RUTA_BASE (str): raiz del proyecto
#   - device (str): 'cuda' (GPU) o 'cpu'
#   - CONFIG (dict): todas las perillas juntas
#     - BATCH_SIZE: imagenes por lote
#     - EPOCHS: cuantas veces ver el dataset completo
#     - LR: learning rate (que tan rapido aprende)
#     - DROPOUT: % de neuronas apagadas al azar
#     - PATIENCE: epocas sin mejora antes de parar
#
# ?Por que batch_size=32?
#   En binario usamos 16 porque ResNet18 pesa ~2.5GB.
#   Para CNN desde cero, el modelo es mas liviano
#   y podemos usar 32 sin problemas.
#
# ?Por que lr=0.001?
#   Valor por defecto de Adam. Funciona en ~80% de los casos.
#
# ?Para experimentar:
#   - Si el kernel muere (OOM), baja BATCH_SIZE a 16 o 8
#   - Si la loss no baja, prueba lr=0.0005
# ============================================

import os, sys, time, json, warnings, math, gc
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import torchvision.models as models

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')

AUTORA = "Diana Blanco - MorritaConP1to"
INICIO = time.time()

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/Diplomado/Modulo4/Proyectos/TADC_Clasificador'
else:
    RUTA_BASE = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

RUTA_DATASET = os.path.join(RUTA_BASE, 'dataset')
RUTA_MODELOS = os.path.join(RUTA_BASE, 'modelos')
os.makedirs(RUTA_MODELOS, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
print('Dataset:', RUTA_DATASET)

CONFIG = {
    'BATCH_SIZE': 32,
    'EPOCHS': 250,  # Early stopping detiene antes; este es el maximo
    'LR': 0.001,
    'DROPOUT': 0.5,
    'PATIENCE': 7,
    'IMG_SIZE': 224,
    'AUTORA': AUTORA,
}
print('CONFIG:', CONFIG)


In [ ]:
# (Ajuste: batch_size menor para ResNet18)
CONFIG['BATCH_SIZE'] = 16
CONFIG['EPOCHS_FASE1'] = 10
CONFIG['EPOCHS_FASE2'] = 15
CONFIG['LR_FASE1'] = 0.001
CONFIG['LR_FASE2'] = 0.0001
CONFIG['DROPOUT'] = 0.3
CONFIG['PATIENCE'] = 5
print('CONFIG (ajustado para TL):', CONFIG)


In [ ]:
# ============================================
# Historial de experimentos anteriores
# ============================================
# Edita EXPERIMENTO_ACTUAL antes de entrenar para
# registrar QUE estas probando y POR QUE.
#
# Variantes sugeridas:
#   1) BASE: config actual
#   2) LR 0.0005  -> si la loss se estanca
#   3) WD 1e-4    -> regularizacion contra overfitting
#   4) Mas filtros (64->128->256->512) -> mas capacidad
#   5) Combinado: LR bajo + WD + mas filtros
# ============================================

EXPERIMENTO_ACTUAL = {
    'id': 1,
    'desc': 'Base',
    'razon': 'Configuracion inicial',
    'cambios': 'Ninguno',
}

RUTA_EXP = os.path.join(RUTA_MODELOS, 'experimentos_tl.json'.format('{}'))
exp_historial = []

---
## Data Augmentation: estirando el dataset

### Por que?

Con ~1000 imagenes por clase y N clases, la aumentacion es clave
para que el modelo generalice y no memorice fondos o angulos.

### Analogia

Es como estudiar para un examen donde el profesor puede cambiar
el orden de las preguntas, el color del papel y el tipo de letra.
Si practicas siempre con el mismo formato, el dia del examen
cualquier cambio te desconcierta.

### Transformaciones

| Transformacion | Simula |
|---------------|--------|
| RandomResizedCrop(224) | La figura no siempre esta centrada |
| RandomHorizontalFlip | Foto desde el otro lado |
| RandomRotation(15deg) | Camara ligeramente inclinada |
| ColorJitter | Distinta iluminacion |


In [ ]:
# ============================================
# Cargar dataset: augment + ImageFolder
#
# ?Que hace: Crea pipelines de transformacion
#   y carga imagenes con ImageFolder.
#
# ?Variables:
#   - transform_train: con aumentacion (entrenar)
#   - transform_test: sin aumentacion (evaluar)
#   - train_dataset (ImageFolder): asocia cada imagen
#     con su clase segun la carpeta donde esta
#   - clases (list): nombres de las clases detectadas
#   - NUM_CLASES (int): numero de personajes
#   - train_loader (DataLoader): itera en lotes
#
# ?Por que ImageFolder?
#   Toma la estructura de carpetas y asigna
#   automaticamente: carpeta = clase.
#   dataset/train/hello_kitty/ -> clase 0
#   dataset/train/kuromi/ -> clase 1
#   etc. (orden ALFABETICO)
#
# ?Para experimentar:
#   - Agrega o quita carpetas en train/
#     El notebook se adapta solo al NUM_CLASES
# ============================================

transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=os.path.join(RUTA_DATASET, 'train'),
                            transform=transform_train)
test_dataset = ImageFolder(root=os.path.join(RUTA_DATASET, 'test'),
                           transform=transform_test)

clases = train_dataset.classes
NUM_CLASES = len(clases)
print('Clases detectadas:', clases)
print('Total clases:', NUM_CLASES)
print('Train:', len(train_dataset), 'imagenes')
print('Test:', len(test_dataset), 'imagenes')
print('Mapa clase->indice:', train_dataset.class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False)


In [ ]:
# ============================================
# Visualizar muestras por clase
#
# ?Que hace: Muestra 4 imagenes de cada clase
#   para confirmar que los datos se cargaron bien.
#   Las imagenes de train se ven "diferentes"
#   cada vez por la aumentacion aleatoria.
# ============================================

# Limpiar CUDA residual antes de graficar
plt.close('all')
gc.collect()
torch.cuda.empty_cache()

def mostrar_muestras(dataset, clases, num_por_clase=4):
    num_clases = len(clases)
    fig, axes = plt.subplots(num_clases, num_por_clase,
                             figsize=(num_por_clase*2, num_clases*2.5))
    if num_clases == 1:
        axes = axes.reshape(1, -1)

    for idx_clase, clase in enumerate(clases):
        indices = [i for i, (_, label) in enumerate(dataset) if label == idx_clase]
        np.random.shuffle(indices)
        indices = indices[:num_por_clase]

        for j, idx in enumerate(indices):
            img, _ = dataset[idx]
            img = img.numpy().transpose(1, 2, 0)
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img = img * std + mean
            img = np.clip(img, 0, 1)

            axes[idx_clase, j].imshow(img)
            axes[idx_clase, j].axis('off')
            if j == 0:
                axes[idx_clase, j].set_ylabel(clase[:12], fontsize=9,
                                              fontweight='bold')

    plt.suptitle('Muestras por clase (train)', fontsize=14)
    plt.tight_layout()
    plt.show()

mostrar_muestras(train_dataset, clases)


---
## Transfer Learning con ResNet18

### Que es Transfer Learning?

En lugar de entrenar desde cero (pesos aleatorios), tomamos una red
que ya fue entrenada con **1.2 MILLONES de imagenes** (ImageNet)
y la adaptamos a nuestro problema.

### Analogia del chef

> Contratas a un chef que ya sabe cocina francesa.
> Tu quieres que aprenda cocina mexicana.
> No le ensenas desde "pelar una papa" (ya lo sabe).
> Solo le ensenas las recetas nuevas (tus ingredientes).
>
> **Eso es Transfer Learning:**
> - El chef = ResNet18 pre-entrenada en ImageNet
> - Las recetas = nuestras fotos de TADC
> - Pelar papas = detectar bordes, colores, texturas

### Por que ResNet18?

| Modelo | Parametros | Con 1000-2000 imagenes |
|--------|-----------|------------------------|
| ResNet18 | 11.7M | Punto dulce: suficiente capacidad, poco overfitting |
| ResNet34 | 21.8M | Demasiada capacidad para ~1000 img/clase |
| ResNet50 | 25.6M | Solo con 5000+ imagenes por clase |

**Arbol de decision:**
```
?Cuantas imagenes por clase?
    |
< 500  ---> ResNet18 (mucho aumento)
500-2000 -> ResNet18 (estas aqui)
2000+  ---> ResNet34 o EfficientNet
```

### Estrategia de 2 fases

```
Fase 1: Solo la cabeza nueva
  +----------+     La cabeza aprende a interpretar
  | Cabeza   |     los features de ImageNet
  | ENTRENA  |     (512 -> 256 -> N_CLASES)
  +----------+
  | layer4   | CONGELADO
  | layer1-3 | CONGELADO (bordes, texturas: universales)

Fase 2: Fine-tuning de layer4
  +----------+
  | Cabeza   | ENTRENA (lr bajo)
  +----------+
  | layer4   | ENTRENA (lr 10x menor)
  +----------+     Ajusta detalles finos de TADC
  | layer1-3 | CONGELADO
```


In [ ]:
# ============================================
# Crear ResNet18 con Transfer Learning
#
# ?Que hace: Carga ResNet18 pre-entrenada, congela
#   todo, y reemplaza la cabeza por una nueva
#   que clasifica NUM_CLASES en vez de 1000.
#
# ?Variables:
#   - model.fc.in_features = 512 (entrada a la FC)
#   - model.fc: la cabeza clasificadora original
#   - requires_grad = False: congela pesos
#
# ?Por que 512 -> 256 -> N_CLASES?
#   Podriamos ir directo 512 -> N_CLASES, pero la
#   capa intermedia de 256 con ReLU y dropout
#   le da mas capacidad a la cabeza sin ser enorme.
#
# ?Para experimentar:
#   - Cambia a models.resnet34(weights=...)
#     ?Cuantos parametros mas tiene?
#   - Prueba sin la capa intermedia (512 -> N_CLASES)
#     ?Aprende igual o peor?
# ============================================

def crear_modelo(num_clases, congelar=True):
    modelo = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if congelar:
        for param in modelo.parameters():
            param.requires_grad = False

    num_features = modelo.fc.in_features

    modelo.fc = nn.Sequential(
        nn.Dropout(CONFIG['DROPOUT']),
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, num_clases),
    )

    return modelo

model = crear_modelo(num_clases=NUM_CLASES, congelar=True).to(device)

total = sum(p.numel() for p in model.parameters())
entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('ResNet18 para {:d} clases'.format(NUM_CLASES))
print('Parametros: {:,} total | {:,} entrenables ({:.1f}%)'.format(
    total, entrenables, 100*entrenables/total))


---
## Dos fases de entrenamiento

### Por que no entrenar todo desde el principio?

| Si entrenas todo desde epoca 1 | Pasa esto |
|------------------------------|-----------|
| La cabeza nueva (pesos aleatorios) | Produce gradientes enormes |
| Esos gradientes "contaminan" layer4 | El modelo olvida ImageNet |
| Resultado | Overfitting inmediato |

### La secuencia ideal

**Fase 1 (10 epocas, lr=0.001):**
Solo la cabeza aprende a interpretar los features de ImageNet.
Loss baja rapido (~0.5 -> 0.1), accuracy sube a ~90-95%.

**Fase 2 (max 15 epocas, lr=0.0001):**
Descongelamos layer4 con learning rate 10x menor.
Ajusta detalles finos de los personajes TADC.
Accuracy puede llegar a 95-99%.

### Early Stopping

Si la loss de test no mejora en 5 epocas, paramos.
No tiene sentido seguir si ya no hay progreso.


In [ ]:
# ============================================
# Optimizadores para cada fase
# ============================================

criterion = nn.CrossEntropyLoss()

optimizer_fase1 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['LR_FASE1']
)

scheduler_fase1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fase1, mode='min', factor=0.5, patience=3
)

print('Fase 1 lista: solo cabeza (lr={})'.format(CONFIG['LR_FASE1']))
print('Fase 2 (despues): fine-tuning layer4 (lr={})'.format(CONFIG['LR_FASE2']))


In [ ]:
# ============================================
# Funcion de entrenamiento (reutilizable)
# ============================================

def entrenar(modelo, train_loader, test_loader, criterion,
             optimizer, scheduler, num_epochs, device, nombre='Fase'):
    historial = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    mejor_loss = float('inf')
    patience = 0

    for epoch in range(num_epochs):
        modelo.train()
        train_loss = 0.0

        for imagenes, etiquetas in train_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            optimizer.zero_grad()
            predicciones = modelo(imagenes)
            loss = criterion(predicciones, etiquetas)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        historial['train_loss'].append(train_loss)

        modelo.eval()
        test_loss = 0.0
        correctos = 0
        total = 0

        with torch.no_grad():
            for imagenes, etiquetas in test_loader:
                imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
                outputs = modelo(imagenes)
                loss = criterion(outputs, etiquetas)
                test_loss += loss.item()
                _, predichos = torch.max(outputs, 1)
                total += etiquetas.size(0)
                correctos += (predichos == etiquetas).sum().item()

        test_loss /= len(test_loader)
        accuracy = 100.0 * correctos / total
        historial['test_loss'].append(test_loss)
        historial['test_acc'].append(accuracy)

        lr_actual = optimizer.param_groups[0]['lr']
        print('{} [{:2d}/{}]  Train Loss: {:.4f}  Test Loss: {:.4f}  Acc: {:.2f}%  LR: {:.6f}'.format(
              nombre, epoch+1, num_epochs, train_loss, test_loss, accuracy, lr_actual))

        scheduler.step(test_loss)

        if test_loss < mejor_loss:
            mejor_loss = test_loss
            torch.save(modelo.state_dict(),
                       os.path.join(RUTA_MODELOS, 'mejor_tl.pth'))
            patience = 0
        else:
            patience += 1
            if patience >= CONFIG['PATIENCE']:
                print('Early stopping en epoca {}'.format(epoch+1))
                break

    print('{} completada. Mejor accuracy: {:.2f}%'.format(nombre, max(historial['test_acc'])))
    return historial


In [ ]:
# ============================================
# Ejecutar Fase 1 + Fase 2
# ============================================

print('=' * 60)
print('FASE 1: ENTRENANDO CABEZA')
print('=' * 60)

historial1 = entrenar(
    model, train_loader, test_loader, criterion,
    optimizer_fase1, scheduler_fase1, CONFIG['EPOCHS_FASE1'],
    device, 'Fase1'
)

print('\n' + '=' * 60)
print('FASE 2: FINE-TUNING (layer4 descongelado)')
print('=' * 60)

for param in model.layer4.parameters():
    param.requires_grad = True

optimizer_fase2 = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': CONFIG['LR_FASE2']},
    {'params': model.fc.parameters(), 'lr': CONFIG['LR_FASE2']}
])

scheduler_fase2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fase2, mode='min', factor=0.5, patience=3
)

historial2 = entrenar(
    model, train_loader, test_loader, criterion,
    optimizer_fase2, scheduler_fase2, CONFIG['EPOCHS_FASE2'],
    device, 'Fase2'
)

historial = {
    'train_loss': historial1['train_loss'] + historial2['train_loss'],
    'test_loss': historial1['test_loss'] + historial2['test_loss'],
    'test_acc': historial1['test_acc'] + historial2['test_acc']
}

print('\nEntrenamiento completado.')
acc_inicial = max(historial['test_acc'])
print('Mejor accuracy: {:.2f}%'.format(acc_inicial))


In [ ]:
# ============================================
# Guardar resultado en historial de experimentos
# ============================================

from datetime import datetime

nuevo_exp = {
    'id': EXPERIMENTO_ACTUAL['id'],
    'descripcion': EXPERIMENTO_ACTUAL['desc'],
    'razon': EXPERIMENTO_ACTUAL['razon'],
    'cambios': EXPERIMENTO_ACTUAL['cambios'],
    'accuracy': max(historial['test_acc']),
    'final_acc': historial['test_acc'][-1],
    'epochs': len(historial['test_acc']),
    'config': {k: v for k, v in CONFIG.items()
               if k not in ('AUTORA',)},
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M'),
}

if not any(e['id'] == nuevo_exp['id'] for e in exp_historial):
    exp_historial.append(nuevo_exp)
    with open(RUTA_EXP, 'w') as f:
        json.dump(exp_historial, f, indent=2)
    print('Experimento #{} guardado: {:.2f}%'.format(
        nuevo_exp['id'], nuevo_exp['accuracy']))
else:
    print('Experimento #{} ya existe, no se sobreescribe'.format(
        nuevo_exp['id']))


---
## Visualizando el progreso

### Que muestra cada grafica

| Grafica | Que esperar |
|---------|-------------|
| Loss | Azul (train) y roja (test) bajan juntas |
| Accuracy | Sube y se estabiliza |
| Comparacion | Fase 2 debe superar Fase 1 |
| Accuracy final | El numero grande |

La linea punteada separa Fase 1 (izquierda) de Fase 2 (derecha).
Nota el "salto" al empezar Fase 2.


In [ ]:
# ============================================
# Graficas con separacion de fases
# ============================================

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 8))

sep = len(historial1['train_loss'])

ax1.plot(range(1, len(historial['train_loss'])+1), historial['train_loss'],
         'b-', label='Train Loss', linewidth=2)
ax1.plot(range(1, len(historial['test_loss'])+1), historial['test_loss'],
         'r-', label='Test Loss', linewidth=2)
ax1.axvline(x=sep, color='gray', linestyle='--', alpha=0.5, label='Fine-tuning')
ax1.set_title('Perdida (Loss)')
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(historial['test_acc'])+1), historial['test_acc'],
         'g-', linewidth=2)
ax2.axvline(x=sep, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Accuracy en Test')
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3)

ax3.bar(['Fase 1', 'Fase 2'],
        [max(historial1['test_acc']), max(historial2['test_acc'])],
        color=['steelblue', 'coral'])
ax3.set_title('Mejor Accuracy por Fase')
ax3.set_ylabel('Accuracy (%)')
ax3.set_ylim(0, 105)

ultimo_acc = historial['test_acc'][-1]
ax4.text(0.5, 0.6, '{:.2f}%'.format(ultimo_acc),
         fontsize=42, ha='center', fontweight='bold', color='green')
ax4.text(0.5, 0.3, 'Accuracy Final',
         fontsize=16, ha='center', color='gray')
ax4.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# Tabla comparativa de todos los experimentos
# ============================================

print('=== COMPARATIVA DE EXPERIMENTOS ===')
print('  {:>3s}  {:25s}  {:30s}  {:>6s}  {:>6s}'.format(
    '#', 'Descripcion', 'Cambio', 'Acc max', 'Epocas'))
print('  ' + '-' * 77)

if exp_historial:
    for e in sorted(exp_historial, key=lambda x: x['accuracy'], reverse=True):
        print('  {:3d}  {:25s}  {:30s}  {:5.1f}%  {:4d}'.format(
            e['id'], e['descripcion'],
            e.get('cambios', e['razon'])[:30],
            e['accuracy'], e['epochs']))

    mejor = max(exp_historial, key=lambda x: x['accuracy'])
    print()
    print('  MEJOR EXPERIMENTO: #{} - {} con {:.2f}%'.format(
        mejor['id'], mejor['descripcion'], mejor['accuracy']))
else:
    print('  (No hay datos)')


---
## Leyendo las metricas

### Matriz de confusion multiclase

La diagonal principal son los aciertos. Fuera de la diagonal son errores.
Con N clases, la matriz es N x N.

**Ejemplo de lectura:**
| Real -> | Predijo A | Predijo B | Predijo C |
|---------|-----------|-----------|-----------|
| **Era A** | 50 (bien) | 2 | 1 |
| **Era B** | 3 | 45 (bien) | 5 |
| **Era C** | 1 | 2 | 48 (bien) |

### Accuracy por clase

Mide que tan bien clasifica CADA personaje individualmente.
Si una clase tiene accuracy bajo, esa figura necesita mas datos
o fotos mas variadas.

### Que esperar

| Accuracy | Significado |
|----------|-------------|
| ~30-50% | Aleatorio (N=12, azar = 8.3%) |
| 60-70% | CNN desde cero, aprendizaje basico |
| 70-85% | CNN decente |
| 85-95% | CNN buena (con aumentacion) |
| 95%+ | Transfer Learning |


In [ ]:
# ============================================
# Evaluacion: matriz de confusion + accuracy por clase
#
# ?Que hace: Pasa todas las imagenes de test por
#   el modelo y calcula metricas detalladas.
#
# ?Por que matriz de confusion?
#   El accuracy total puede esconder problemas:
#   - Una clase con muchos errores
#   - El modelo confunde sistematicamente 2 clases
#   - La matriz revela estos patrones
# ============================================

# Limpiar CUDA residual antes de evaluar
plt.close('all')
gc.collect()
torch.cuda.empty_cache()

def evaluar(modelo, test_loader, device, clases):
    modelo.eval()
    todas_reales = []
    todas_predichas = []

    with torch.no_grad():
        for imagenes, etiquetas in test_loader:
            imagenes = imagenes.to(device)
            outputs = modelo(imagenes)
            _, predichas = torch.max(outputs, 1)

            todas_reales.extend(etiquetas.cpu().numpy())
            todas_predichas.extend(predichas.cpu().numpy())

    cm = confusion_matrix(todas_reales, todas_predichas)

    fig, ax = plt.subplots(1, 1, figsize=(max(8, len(clases)*0.8),
                                          max(6, len(clases)*0.7)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clases)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d',
              xticks_rotation=45)
    ax.set_title('Matriz de Confusion Multiclase', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\n=== Accuracy por Clase ===')
    aciertos_total = 0
    for i, clase in enumerate(clases):
        total_clase = cm[i, :].sum()
        correctos = cm[i, i]
        acc = 100.0 * correctos / total_clase if total_clase > 0 else 0
        aciertos_total += correctos
        barra = '?' * int(acc / 5) + '?' * (20 - int(acc / 5))
        print('  {:<20s} {:>3d}/{:>3d} = {:>5.1f}% {}'.format(
            clase, correctos, total_clase, acc, barra))

    accuracy = 100.0 * aciertos_total / cm.sum()
    print('\n  Accuracy TOTAL: {:.2f}%  ({:d}/{:d} aciertos)'.format(
        accuracy, aciertos_total, cm.sum()))
    return cm

cm = evaluar(model, test_loader, device, clases)


---
## Donde falla? Aprendiendo de los errores

Los errores revelan las debilidades del modelo. Esta celda
muestra las imagenes mal clasificadas.

### Que buscar

| Patron de error | Posible causa | Solucion |
|----------------|--------------|----------|
| Confunde siempre 2 clases | Son muy parecidas visualmente | Mas datos de ambas, o aumentacion especifica |
| Errores en una sola clase | Pocas imagenes de esa clase en test | Balancear dataset |
| Errores con fondo similar | El modelo aprendio el fondo, no la figura | Variar fondos en las fotos |


In [ ]:
# ============================================
# Grilla de errores
# ============================================

plt.close('all')
gc.collect()
torch.cuda.empty_cache()

model.eval()
imagenes_guardadas = []
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes = imagenes.to(device)
        outputs = model(imagenes)
        probabilidades = torch.softmax(outputs, dim=1)
        _, predichas = torch.max(outputs, 1)
        for i in range(len(imagenes)):
            if len(imagenes_guardadas) < 200:
                imagenes_guardadas.append((
                    imagenes[i].cpu(), etiquetas[i].item(),
                    predichas[i].item(), probabilidades[i]))

errores = [(img, r, p, probs) for (img, r, p, probs) in imagenes_guardadas if r != p]

if len(errores) == 0:
    print('\n? No hubo errores en las 200 muestras guardadas!')
else:
    n_mostrar = min(len(errores), 12)
    cols = 4
    rows = math.ceil(n_mostrar / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3.5, rows*3.5))
    axes = axes.flatten() if n_mostrar > 1 else [axes]

    for i in range(n_mostrar):
        img, real, pred, probs = errores[i]
        img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) +               torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        img = torch.clamp(img, 0, 1)
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        conf = probs[pred].item() * 100
        axes[i].set_title('Real: {}\nPred: {} ({:.0f}%)'.format(
            clases[real], clases[pred], conf),
            fontsize=9, color='red')
        axes[i].axis('off')

    for i in range(n_mostrar, len(axes)):
        axes[i].axis('off')

    plt.suptitle('Errores ({}/{} muestras)'.format(
        len(errores), len(imagenes_guardadas)),
        fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\nTasa de error: {:.1f}%'.format(
        100 * len(errores) / len(imagenes_guardadas)))


---
## Probando con tus propias fotos

Toma una foto de un personaje TADC y el modelo te dira cual es.
Muestra el Top-3 con barras de confianza.


In [ ]:
# ============================================
# Prediccion batch desde carpeta
#
# ?Que hace: Escanea RUTA_PRUEBAS por imagenes,
#   corre prediccion en cada una, y muestra
#   una grilla con el resultado.
#
# Cambia RUTA_PRUEBAS si tus imagenes estan en
# otra carpeta.
# ============================================

RUTA_PRUEBAS = 'modelos'  # Cambia esta ruta si gustas

EXTENSIONES = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

def predecir_imagen(ruta_imagen, modelo, device, clases):
    modelo.eval()
    imagen = Image.open(ruta_imagen).convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    tensor = transform(imagen).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = modelo(tensor)
        probabilidades = torch.softmax(outputs, dim=1)[0]

    prob_np = probabilidades.cpu().numpy()
    idx_pred = int(torch.argmax(probabilidades))
    conf = prob_np[idx_pred] * 100
    return imagen, clases[idx_pred], conf, prob_np

if EN_COLAB:
    from google.colab import files
    subidos = files.upload()
    archivos_prueba = list(subidos.keys())
    RUTA_PRUEBAS = None  # No usamos carpeta en Colab
else:
    if os.path.isdir(RUTA_PRUEBAS):
        archivos_prueba = [os.path.join(RUTA_PRUEBAS, f)
                           for f in os.listdir(RUTA_PRUEBAS)
                           if f.lower().endswith(EXTENSIONES)]
        archivos_prueba.sort()
    else:
        archivos_prueba = []
        print('? La carpeta "%s" no existe.' % RUTA_PRUEBAS)

if not archivos_prueba:
    print('No se encontraron imagenes en %s.' % RUTA_PRUEBAS)
else:
    n = len(archivos_prueba)
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten() if n > 1 else [axes]

    for i, ruta in enumerate(archivos_prueba):
        nombre = os.path.basename(ruta)
        try:
            img, pred, conf, _ = predecir_imagen(ruta, model, device, clases)
            axes[i].imshow(img)
            axes[i].set_title('%s\n? %s (%.1f%%)' % (nombre, pred, conf),
                              fontsize=10, fontweight='bold', color='green')
        except Exception as e:
            axes[i].set_title('%s\n? Error' % nombre, fontsize=10, color='red')
        axes[i].axis('off')

    for i in range(n, len(axes)):
        axes[i].axis('off')

    plt.suptitle('Predicciones del modelo', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================
# Guardar modelo Transfer Learning
# ============================================

from datetime import datetime

NOMBRE_MODELO = 'tl_multiclase'
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

torch.save(model.state_dict(),
           os.path.join(RUTA_MODELOS, NOMBRE_MODELO + '_{}.pth'.format(timestamp)))
torch.save(model.state_dict(),
           os.path.join(RUTA_MODELOS, NOMBRE_MODELO + '_final.pth'))

with open(os.path.join(RUTA_MODELOS, 'clases_multiclase.json'), 'w') as f:
    json.dump({
        'classes': clases,
        'class_to_idx': train_dataset.class_to_idx,
        'num_classes': NUM_CLASES,
        'fecha': timestamp,
        'autora': AUTORA,
    }, f, indent=2)

print('Modelo Transfer Learning guardado.')
print('Clases:', clases)


In [ ]:
# ============================================
# Footer: autoria y rendimiento
# ============================================

import psutil, platform

fin = time.time()
segundos = fin - INICIO
mm, ss = divmod(int(segundos), 60)

print('')
print('=' * 55)
print('  Hecho con carino por ' + AUTORA)
print('  Diplomado RNA - Modulo 4: Deep Learning')
print('  UAEM')
print('')
print('  Especificaciones:')
cpu = platform.processor() or 'AMD Ryzen'
print('    CPU: ' + cpu)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
print('    GPU: ' + gpu_name + ' (' + str(gpu_mem) + ' GB)')
ram = round(psutil.virtual_memory().total / 1e9, 1)
print('    RAM: ' + str(ram) + ' GB')
print('    PyTorch ' + torch.__version__)
print('')
print('  Tiempo: {:d} min {:d} seg'.format(mm, ss))
print('  Accuracy: {:.2f}%'.format(historial['test_acc'][-1]))
print('  Clases: {:d}'.format(NUM_CLASES))
print('  Fecha: ' + time.strftime('%Y-%m-%d %H:%M'))
print('=' * 55)
